<a href="https://colab.research.google.com/github/bavucadoz/244107020162_ML_2026/blob/main/Minggu06/P03_Week06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Praktikum 3**
*Percobaan berikut akan membandingkan exact NN dengan HNSW pada 1000 data 2D.*


---



In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=22f7925f9624b07c796bb2cfb1dabdfe1be7662e474d1f9dca2a75fae7e4d143
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[886  29 960 144 827]]
Distances: [[0.01506602 0.02462637 0.02762381 0.02959388 0.04231765]]
Waktu: 0.0292665958404541 detik

=== HNSW ===
Indices: [[886  29 960 144 827]]
Distances: [[0.00022699 0.00060646 0.00076308 0.0008758  0.00179078]]
Waktu: 0.0001442432403564453 detik




---
**Tabel Percobaan**


In [3]:
import hnswlib
import numpy as np
import time
import pandas as pd
from sklearn.neighbors import NearestNeighbors

def run_hnsw_experiment(metric_name, n_points, dim):
    # Set seed agar hasil random konsisten
    np.random.seed(42)

    # Generate dataset & query
    data = np.random.random((n_points, dim)).astype(np.float32)
    query = np.random.random((1, dim)).astype(np.float32)
    k = 5  # cari 5 tetangga terdekat

    # 1. Persiapan Metric
    if metric_name == 'Euclidean':
        sk_metric = 'euclidean'
        hnsw_space = 'l2'
    elif metric_name == 'Cosine':
        sk_metric = 'cosine'
        hnsw_space = 'cosine'

    # 2. Exact NN (Brute Force dengan Sklearn)
    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=sk_metric)
    nn.fit(data)

    start_exact = time.perf_counter()
    distances_exact, indices_exact = nn.kneighbors(query)
    time_exact = (time.perf_counter() - start_exact) * 1000  # konversi ke milidetik (ms)

    # 3. Approximate NN (HNSW)
    p = hnswlib.Index(space=hnsw_space, dim=dim)

    # M: jumlah koneksi bi-directional yang dibuat per node
    # ef_construction: ukuran antrian dinamis saat membuat index (tradeoff waktu build vs kualitas)
    p.init_index(max_elements=n_points, ef_construction=100, M=16)
    p.add_items(data)

    # ef: ukuran antrian dinamis saat melakukan query pencarian (tradeoff waktu query vs recall)
    p.set_ef(50)

    start_hnsw = time.perf_counter()
    labels_hnsw, distances_hnsw = p.knn_query(query, k=k)
    time_hnsw = (time.perf_counter() - start_hnsw) * 1000  # konversi ke milidetik (ms)

    # Format Output Tabel
    fmt_idx_exact = f"[{' '.join(map(str, indices_exact[0]))}]"
    fmt_idx_hnsw = f"[{' '.join(map(str, labels_hnsw[0]))}]"

    return {
        "Metrik Jarak": metric_name,
        "Dimensi": f"{dim}D",
        "Jumlah Data": f"{n_points:,}",
        "Index Terdekat (ENN vs HNSW)": f"{fmt_idx_exact} , {fmt_idx_hnsw}",
        "Waktu ms (Vs)": f"{time_exact:.4f} , {time_hnsw:.4f}"
    }

scenarios = [
    ('Euclidean', 1000, 2),
    ('Euclidean', 1000, 5),
    ('Euclidean', 1000000, 2),
    ('Euclidean', 1000000, 5),
    ('Cosine', 1000, 2),
    ('Cosine', 1000, 5),
    ('Cosine', 1000000, 2),
    ('Cosine', 1000000, 5)
]

results = [run_hnsw_experiment(m, n, d) for m, n, d in scenarios]

df = pd.DataFrame(results)
print("\n")
print(df.to_markdown(index=False, stralign='left'))



| Metrik Jarak   | Dimensi   | Jumlah Data   | Index Terdekat (ENN vs HNSW)                                                | Waktu ms (Vs)    |
|:---------------|:----------|:--------------|:----------------------------------------------------------------------------|:-----------------|
| Euclidean      | 2D        | 1,000         | [112 535 777 246 763] , [112 535 777 246 763]                               | 3.1132 , 0.0616  |
| Euclidean      | 5D        | 1,000         | [988 780 27 943 93] , [988 780 27 943 93]                                   | 0.6003 , 0.0666  |
| Euclidean      | 2D        | 1,000,000     | [132774 119034 511191 482199 913140] , [132774 119034 511191 482199 913140] | 22.6261 , 0.0901 |
| Euclidean      | 5D        | 1,000,000     | [901095 495168 561017 163986 185734] , [901095 495168 561017 163986 185734] | 26.1479 , 0.1374 |
| Cosine         | 2D        | 1,000         | [738 183 561 91 244] , [738 183 561 91 244]                                 | 18.5963 ,